# Chapter 5 — nn.Module

**Book alignment:** PyTorch From First Principles, Chapter 5

**Question this notebook isolates:** Does holding submodules in a plain Python list (vs `nn.ModuleList`) exclude them from `parameters()`, `state_dict()`, and optimizer updates while leaving the forward computation intact?


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)


## 1. Registration census: the plain list hides three layers

Same architecture twice — once with `self.blocks = [...]`, once with `nn.ModuleList`. The forward math is identical; the parameter counts must differ by the hidden layers.


In [ ]:
class PlainEncoder(nn.Module):
    def __init__(self, width=8, depth=2):
        super().__init__()
        self.blocks = [nn.Linear(width, width) for _ in range(depth)]
    def forward(self, x):
        for block in self.blocks:
            x = torch.relu(block(x))
        return x

class ListedEncoder(nn.Module):
    def __init__(self, width=8, depth=2):
        super().__init__()
        self.blocks = nn.ModuleList([nn.Linear(width, width) for _ in range(depth)])
    def forward(self, x):
        for block in self.blocks:
            x = torch.relu(block(x))
        return x

class Net(nn.Module):
    def __init__(self, enc):
        super().__init__()
        self.stem = nn.Linear(2, 8)
        self.encoder = enc
        self.head = nn.Linear(8, 2)
    def forward(self, x):
        return self.head(self.encoder(torch.relu(self.stem(x))))

torch.manual_seed(0)
plain = Net(PlainEncoder())
torch.manual_seed(0)
listed = Net(ListedEncoder())
for name, m in (("plain", plain), ("listed", listed)):
    ps = list(m.parameters())
    print(f"{name}: tensors={len(ps)} numel={sum(p.numel() for p in ps)}")
print("plain keys:", sorted(plain.state_dict()))


In [ ]:
assert len(list(plain.parameters())) == 4
assert sum(p.numel() for p in plain.parameters()) == 42
assert len(list(listed.parameters())) == 8
assert sum(p.numel() for p in listed.parameters()) == 42 + 2 * (8 * 8 + 8)
assert not any(k.startswith("encoder.blocks") for k in plain.state_dict())
assert any(k.startswith("encoder.blocks") for k in listed.state_dict())
print("identical math, different ownership: 144 numbers missing from plain")


## 2. Checkpoint asymmetry: a clean load that proves nothing

A `state_dict` saved from the plain model must load into a fresh plain model with zero missing keys — while proving nothing about the hidden layers, which were never saved.


In [ ]:
ckpt = plain.state_dict()
fresh = Net(PlainEncoder())
missing, unexpected = fresh.load_state_dict(ckpt, strict=False), None
print(f"load into fresh plain model: missing={missing.missing_keys} unexpected={missing.unexpected_keys}")
full = Net(ListedEncoder())
res = full.load_state_dict(ckpt, strict=False)
print(f"plain ckpt into listed model: {len(res.missing_keys)} missing keys")
print(f"example: {sorted(res.missing_keys)[0]}")


In [ ]:
assert missing.missing_keys == [] and missing.unexpected_keys == []
assert len(res.missing_keys) == 4, res.missing_keys
assert all(k.startswith("encoder.blocks") for k in res.missing_keys)
print("<All keys matched> can mean the checkpoint never contained the encoder")


## 3. Optimizer membership: hidden layers train in one model, freeze in the other

One SGD step on each model. The listed encoder's weights must move; the plain encoder's must be bit-identical (the optimizer never saw them).


In [ ]:
torch.manual_seed(1)
xb = torch.randn(32, 2)
yb = torch.randint(0, 2, (32,))

def one_step(model):
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    before = [p.detach().clone() for p in model.parameters()]
    enc_before = model.encoder.blocks[0].weight.detach().clone()
    opt.zero_grad()
    torch.nn.functional.cross_entropy(model(xb), yb).backward()
    opt.step()
    moved = max((p.detach() - o).abs().max().item() for p, o in zip(model.parameters(), before))
    enc_moved = (model.encoder.blocks[0].weight.detach() - enc_before).abs().max().item()
    return moved, enc_moved

torch.manual_seed(0)
m_plain = Net(PlainEncoder())
torch.manual_seed(0)
m_listed = Net(ListedEncoder())
mp, ep = one_step(m_plain)
ml, el = one_step(m_listed)
print(f"plain:  any-param moved={mp:.5f} encoder moved={ep:.5f}")
print(f"listed: any-param moved={ml:.5f} encoder moved={el:.5f}")


In [ ]:
assert mp > 0, "plain stem/head still train"
assert ep == 0.0, "plain encoder untouched: not in optimizer"
assert el > 0, "listed encoder trains"
print("optimizer updates membership, not reachability")


## What we earned

Four structures can disagree: Python reachability (forward runs), registration (`parameters`/`state_dict`), autograd (gradients flow), optimizer membership (weights move). Check each directly instead of inferring ownership from source.

Chapter 6 leaves the model fixed and asks where the training loop actually waits: the input pipeline as a producer–consumer system.
